In [ ]:
import torch
from matplotlib import pyplot as plt

In [ ]:
from diffusion_linking.diffusion import VariationalDiffusion
from diffusion_linking.data import generate_circles
from diffusion_linking.clustering import Clusterer
from diffusion_linking.training import train_model
from diffusion_linking.schedule import MonotoneNetSchedule

In [ ]:
circles, masks = generate_circles(100_000, min_points=1, max_points=20, min_radius=0.6, max_radius=0.6)
schedule = MonotoneNetSchedule(20, epsilon=1e-4)
# schedule = LinearSchedule()
model = VariationalDiffusion(dim=2, depth=12, schedule=schedule).to('cuda')
dataloader = torch.utils.data.DataLoader(list(zip(circles, masks)), batch_size=64, shuffle=True)

In [ ]:
# plot the noise schedule
xx = torch.linspace(0, 1, 100)
plt.plot(xx, model.schedule(xx.cuda()).detach().cpu().numpy())

In [ ]:
# load a checkpoint if one exists, else train the model from scratch
checkpoint_path = "circles_scheduled.pt"
try:
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model'])
    loss_history = checkpoint['loss_history']
    model = model.to('cuda')
    print('Loaded checkpoint')
except:
    print('Training model from scratch')
    loss_history, model = train_model(model, dataloader, epochs=50, checkpoint_path=checkpoint_path, loss_interval=100)
    plt.plot(loss_history)
    plt.semilogy()

In [ ]:
# train further
loss_history, model = train_model(model, dataloader, epochs=100, checkpoint_path=checkpoint_path, loss_interval=100)

In [ ]:
plt.plot(loss_history)
plt.semilogy()

In [ ]:
# plot the noise schedule
xx = torch.linspace(0, 1, 100)
plt.plot(xx, model.schedule(xx.cuda()).detach().cpu().numpy())

In [ ]:
# plot some samples fom the model
c = model.generate(num_samples=15, seq_len=10, num_time_steps=128, device='cuda').cpu().detach().numpy()
for cc in c:
    plt.plot(cc[:, 0], cc[:, 1], 'o', markersize=3)
plt.axis('equal')

In [ ]:
# plot some samples fom the model using ODE
c = model.generate_ode(num_samples=15, seq_len=10, num_time_steps=1024, device='cuda',
                       masks=None).cpu().detach().numpy()
for cc in c:
    plt.plot(cc[:, 0], cc[:, 1], 'o', markersize=3)
plt.axis('equal')

In [ ]:
for i in range(5):
    logp = model.logp_smc(circles[i][masks[i]].unsqueeze(0).to('cuda'), num_particles=1024, num_time_steps=128)[0]
    print(logp)

In [ ]:
logp, xT = model.log_prob_ode(
    torch.stack(circles[:5]).to('cuda'), torch.stack(masks[:5]).to('cuda'),
    num_time_steps=128)
logp

In [ ]:
for xti in xT:
    plt.plot(xti[:, 0].cpu().detach().numpy(), xti[:, 1].cpu().detach().numpy(), 'o', markersize=3)

regenerated = model.generate_ode(
    num_samples=5, seq_len=10, num_time_steps=512, device='cuda',
    masks=torch.stack(masks[:5]).to('cuda'), z1=xT).cpu().detach().numpy()
plt.figure()
for c in regenerated:
    plt.plot(c[:, 0], c[:, 1], 'o', markersize=3)
plt.gca().set_prop_cycle(None)
for c in circles[:5]:
    plt.plot(c[:, 0], c[:, 1], 'x')
plt.axis('equal')

In [ ]:
# make a new circles dataset and see if we can link it
circles_valid, masks_valid = generate_circles(3, min_points=19, max_points=20, min_radius=0.6, max_radius=0.6)
for cc in circles_valid:
    plt.plot(cc[:, 0], cc[:, 1], 'o', markersize=3)
plt.axis('equal')

In [ ]:
# build clusters from our toy linking data
cluster_data = []
for c, m in zip(circles_valid, masks_valid):
    for ci, mi in zip(c, m):
        if mi:
            cluster_data.append(ci.unsqueeze(0))


def plot_callback(clusters, iteration):
    # sort clusters by size
    clusters = sorted(clusters, key=lambda c: c.size, reverse=True)
    if iteration % 10 == 0:
        plt.figure()
        for c in clusters:
            if c.size > 1:
                plt.plot(c.data[:, 0].cpu().numpy(), c.data[:, 1].cpu().numpy(), 'o', markersize=3)
            else:
                plt.plot(c.data[:, 0].cpu().numpy(), c.data[:, 1].cpu().numpy(), 'ks', markersize=3, alpha=0.5)
        plt.axis('equal')

    print(
        f"iteration {iteration}, {len(clusters)} clusters, {sum([c.size for c in clusters])} points, {[c.size for c in clusters]}")


num_time_steps = 40


def score_func(data, masks):
    scores, _ = model.log_prob_ode(data.cuda(), masks.cuda(), num_time_steps=num_time_steps)
    return scores


clusterer = Clusterer(data=cluster_data, score_fn=score_func)
clusterer.cluster(max_iter=100, verbose=True, callback=plot_callback)
plot_callback(clusterer.clusters, 0)

In [ ]:
plot_callback(clusterer.clusters, 0)